# Εργαστήριο Ακτίνων Χ 2026 — NGC 300 ULX-1: spin-up ενός υπερφωτεινού πάλσαρ ακτίνων Χ

Αυτό το notebook αντιστοιχεί στο **Μέρος 1** των σημειώσεων (Κεφάλαιο 2). Στόχος: να μετρήσουμε τη συχνότητα περιστροφής $\nu$ του αστέρα νετρονίων και πόσο γρήγορα αλλάζει, $\dot\nu$, κατά τη διάρκεια μιας παρατήρησης NuSTAR ~3.5 ημερών.

**Πώς δουλεύουμε**
- Εκτελείτε τα κελιά με τη σειρά (`Shift + Enter`).
- ✏️ **TODO** = αλλάζετε ή γράφετε μία-δύο γραμμές κώδικα.
- ❓ = ερώτηση για σκέψη· οι απαντήσεις σας βοηθούν στην εργασία.
- Οι ενότητες **EXTRA** στο τέλος είναι προαιρετικές.

Στο τέλος θα καταχωρίσετε το αποτέλεσμά σας στο **κοινό Google Sheet** της τάξης.

## 0. Λογισμικό

Χρειαζόμαστε τα `numpy`, `scipy`, `matplotlib` και `astropy`. Αν λείπει κάποιο πακέτο, ανοίξτε terminal, ενεργοποιήστε το περιβάλλον του εργαστηρίου και εγκαταστήστε το από το conda-forge, π.χ.

```
conda activate xray
conda install -c conda-forge numpy scipy matplotlib astropy
```

(Το `stingray` χρησιμοποιείται μόνο στην ενότητα EXTRA: `conda install -c conda-forge stingray`.) Μετά την εγκατάσταση κάντε restart τον kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy.optimize
from astropy.io import fits

plt.rcParams['figure.dpi'] = 100

## 1. Επιλογή αρχείου δεδομένων

Έχουμε **τέσσερα** αρχεία γεγονότων από την ίδια παρατήρηση NuSTAR (Δεκέμβριος 2016):

| κλειδί | ανιχνευτής | βαρυκεντρική διόρθωση | αρχείο |
|---|---|---|---|
| `A_bary`   | FPMA | ναι | `nu30202035002A01_cl_bary.evt.gz` |
| `B_bary`   | FPMB | ναι | `nu30202035002B01_cl_bary.evt.gz` |
| `A_nobary` | FPMA | όχι | `nu30202035002A01_cl.evt.gz` |
| `B_nobary` | FPMB | όχι | `nu30202035002B01_cl.evt.gz` |

Το NuSTAR έχει δύο όμοια τηλεσκόπια με δύο ανιχνευτές, FPMA και FPMB. Στα βαρυκεντρικά αρχεία οι χρόνοι άφιξης των φωτονίων έχουν αναχθεί στο βαρύκεντρο του Ηλιακού Συστήματος (βλ. ενότητα EXTRA).

Κάθε φοιτητής/τρια αναλύει **ένα** αρχείο. Συγκεντρώνοντας τα αποτελέσματα όλων, θα ελέγξουμε αν ο ανιχνευτής ή η βαρυκεντρική διόρθωση αλλάζουν τον μετρούμενο $\dot\nu$.

✏️ **TODO:** ορίστε στο `my_choice` το αρχείο που σας αντιστοιχεί (ή που επιλέξατε).

In [ ]:
files = {
    'A_bary':   'nu30202035002A01_cl_bary.evt.gz',
    'B_bary':   'nu30202035002B01_cl_bary.evt.gz',
    'A_nobary': 'nu30202035002A01_cl.evt.gz',
    'B_nobary': 'nu30202035002B01_cl.evt.gz',
}

my_choice = 'A_bary'     # TODO: ένα από 'A_bary', 'B_bary', 'A_nobary', 'B_nobary'

event_file = files[my_choice]
hdul = fits.open(event_file)
data = hdul[1].data          # ο πίνακας των γεγονότων (επέκταση [1])
hdr = hdul[1].header       # πληροφορίες για την παρατήρηση

print('File:', event_file)
for key in ['TELESCOP', 'OBJECT', 'DATE-OBS', 'TSTART', 'TSTOP', 'TIMEREF']:
    print(f'{key:9s} = {hdr.get(key)}')

❓ `TIMEREF = SOLARSYSTEM` σημαίνει ότι οι χρόνοι είναι βαρυκεντρικοί, `LOCAL` ότι δεν είναι. Συμφωνεί με το αρχείο που επιλέξατε;
Σημειώστε το `DATE-OBS`: θα το χρειαστείτε στο Μέρος 2 για να βρείτε ταυτόχρονες παρατηρήσεις XMM-Newton.

## 2. Τι περιέχει ένα αρχείο γεγονότων;

Ένα αρχείο γεγονότων FITS είναι ένας πίνακας: **μία γραμμή για κάθε φωτόνιο** που ανιχνεύθηκε. Ας δούμε ποιες στήλες έχει.

In [ ]:
print(data.columns.names)
print('number of events:', len(data))

✏️ **TODO:** τυπώστε το περιεχόμενο των στηλών `PI` και `TIME` (π.χ. `print(data['PI'])`), καθώς και την ελάχιστη και μέγιστη τιμή τους (`.min()`, `.max()`).

❓ Τι νομίζετε ότι περιέχει κάθε στήλη;
- `TIME`: χρόνος άφιξης κάθε φωτονίου, σε δευτερόλεπτα από μια ημερομηνία αναφοράς της αποστολής (1 Ιανουαρίου 2010 για το NuSTAR).
- `PI`: αριθμός καναλιού, ανάλογος της ενέργειας του φωτονίου. Για το NuSTAR: $E = 0.04 \times \mathrm{PI} + 1.6$ keV.
- `X`, `Y`: θέση του φωτονίου στον ουρανό, σε pixels.

In [ ]:
# TODO: τυπώστε τις στήλες PI και TIME και τα min / max τους
print(data['PI ....

## 3. Εικόνα και περιοχή της πηγής

Μια εικόνα είναι ένα **δισδιάστατο ιστόγραμμα** των θέσεων `X`, `Y` των φωτονίων: μετράμε πόσα φωτόνια πέφτουν σε κάθε μικρό τετράγωνο (bin).

In [ ]:
x, y = data['X'], data['Y']
cmap = mpl.colormaps['gray'].copy(); cmap.set_bad('black')   # τα κενά bins εμφανίζονται μαύρα
H, xedges, yedges = np.histogram2d(x, y, bins=200, range=[[0, 1000], [0, 1000]])   # 5x5-pixel bins

plt.figure(figsize=(6, 5))
plt.imshow(H.T, origin='lower', extent=[0, 1000, 0, 1000], cmap=cmap, norm=mpl.colors.LogNorm())
plt.colorbar(label='counts per bin')
plt.xlim(300, 700); plt.ylim(300, 700)
plt.xlabel('X (pixel)'); plt.ylabel('Y (pixel)')
plt.show()

✏️ **TODO:** κοιτάξτε την εικόνα και εκτιμήστε **με το μάτι** το κέντρο της λαμπρής πηγής (`xc`, `yc`) και μια ακτίνα `rad` (σε pixels) που περιέχει τα περισσότερα φωτόνιά της. Μπορείτε να μεγεθύνετε αλλάζοντας τα `plt.xlim` / `plt.ylim` στο προηγούμενο κελί. Οι δύο ανιχνευτές (A, B) έχουν λίγο διαφορετικές συντεταγμένες, οπότε μην αντιγράψετε τους αριθμούς συναδέλφου που χρησιμοποιεί τον άλλο ανιχνευτή!

*Παράπλευρη εργασία (προαιρετική):* αφού η εικόνα είναι ιστόγραμμα, το κέντρο είναι περίπου το bin με τα περισσότερα γεγονότα. Υπόδειξη: το `i, j = np.unravel_index(np.argmax(H), H.shape)` δίνει τους δείκτες αυτού του bin, και το κέντρο του είναι `0.5*(xedges[i] + xedges[i+1])`, `0.5*(yedges[j] + yedges[j+1])`.
(Ο σωστός τρόπος θα ήταν η προσαρμογή μιας δισδιάστατης Gaussian (ή της PSF) στην εικόνα· αυτό δεν το κάνουμε στο μάθημα.)

In [ ]:
xc, yc = 500.0, 500.0    # TODO: κέντρο της πηγής, με το μάτι (pixels)
rad = 30.0               # TODO: ακτίνα της περιοχής (pixels)

# Παράπλευρη εργασία (προαιρετική): το πιο φωτεινό bin της εικόνας

In [ ]:
def in_circle(xc, yc, rad, x, y):
    # True για γεγονότα μέσα στον κύκλο, False έξω
    return (x - xc)**2 + (y - yc)**2 <= rad**2

src = in_circle(xc, yc, rad, x, y)
print('events in file:', len(x), '   inside the circle:', src.sum())

plt.figure(figsize=(6, 5))
plt.imshow(H.T, origin='lower', extent=[0, 1000, 0, 1000], cmap=cmap, norm=mpl.colors.LogNorm())
plt.gca().add_patch(plt.Circle((xc, yc), rad, fill=False, color='red', lw=2))
plt.xlim(xc - 150, xc + 150); plt.ylim(yc - 150, yc + 150)
plt.xlabel('X (pixel)'); plt.ylabel('Y (pixel)')
plt.show()

## 4. Επιλογή ενέργειας

Κρατάμε τα φωτόνια μέσα στον κύκλο με ενέργειες από 3 έως 30 keV. Κάτω από 3 keV το NuSTAR δεν είναι ευαίσθητο, και πάνω από ~30 keV η πηγή είναι αμυδρή σε σχέση με το υπόβαθρο.

Αφαιρούμε επίσης τον χρόνο του πρώτου φωτονίου, ώστε οι χρόνοι να ξεκινούν από το μηδέν (σε δευτερόλεπτα).

In [ ]:
E = 0.04 * data['PI'] + 1.6                 # ενέργεια σε keV
Emin, Emax = 3.0, ...TODO
sel = src & (E > Emin) & (E <= Emax)

t_ev = data['TIME'][sel]
t_ev = t_ev - t_ev.min()                  # χρόνος από το πρώτο φωτόνιο (s)
T_obs = t_ev.max()

print('selected photons:', len(t_ev))
print(f'duration of the observation: {T_obs:.0f} s = {T_obs/86400:.2f} days')

## 5. Καμπύλη φωτός

Η καμπύλη φωτός είναι ιστόγραμμα των χρόνων άφιξης: ο αριθμός των φωτονίων σε κάθε χρονικό bin.

✏️ **TODO:** στο τέλος του κελιού προσθέστε **μία γραμμή** που μεγεθύνει στα πρώτα 20 ks (20000 s) της παρατήρησης. *Υπόδειξη:* `plt.xlim(...)`.

❓ Τι βλέπετε; Πόσο διαρκεί ένα κενό και κάθε πότε επαναλαμβάνεται; Το NuSTAR βρίσκεται σε χαμηλή τροχιά γύρω από τη Γη: τι μπορεί να προκαλεί τα κενά;

In [ ]:
binsize = 100.0          # δευτερόλεπτα
edges = np.arange(0, T_obs + binsize, binsize)
lc, _ = np.histogram(t_ev, bins=edges)

plt.figure(figsize=(12, 4))
plt.plot(0.5 * (edges[1:] + edges[:-1]), lc / binsize, drawstyle='steps-mid')
plt.xlabel('time since first photon (s)'); plt.ylabel('count rate (counts/s)')
# TODO: προσθέστε εδώ μία γραμμή για μεγέθυνση στα πρώτα 20 ks
plt.show()

## 6. Αναδίπλωση (folding): το προφίλ του παλμού

**Αναδίπλωση** (epoch folding): για μια δοκιμαστική περίοδο $P$ υπολογίζουμε τη **φάση** κάθε φωτονίου, $\phi = (t/P) \bmod 1$ (αριθμός από 0 έως 1), και φτιάχνουμε ιστόγραμμα των φάσεων με `nbin` bins. Αυτό είναι το μέσο **προφίλ του παλμού**.
- Αν η $P$ είναι κοντά στην πραγματική περίοδο, τα φωτόνια συγκεντρώνονται στις ίδιες φάσεις: βλέπουμε καθαρό παλμό.
- Αν η $P$ είναι λάθος, οι φάσεις είναι τυχαίες: το προφίλ είναι επίπεδο.

Το προφίλ σχεδιάζεται δύο φορές (φάση 0–2) μόνο για να φαίνεται καλύτερα.

**Επίπεδο θορύβου.** Αν το προφίλ ήταν επίπεδο, κάθε bin θα περιείχε κατά μέσο όρο $\bar N = N_{tot}/n_{bin}$ φωτόνια, με τυχαίες (Poisson) διακυμάνσεις περίπου $\sqrt{\bar N}$. Η **γκρι ζώνη** δείχνει το $\bar N \pm \sqrt{\bar N}$. Ένα προφίλ που μένει μέσα στη ζώνη είναι συμβατό με απουσία παλμού (ακόμη και για επίπεδο προφίλ, περίπου το 1/3 των bins πέφτει λίγο έξω από τη ζώνη)· ένας πραγματικός παλμός την ξεπερνά καθαρά.

In [ ]:
def fold(times, period, nbin):
    phase = (times / period) % 1.0
    counts, _ = np.histogram(phase, bins=nbin, range=(0.0, 1.0))
    return counts

def plot_profile(counts, label=None):
    nbin = len(counts)
    phase = (np.arange(2 * nbin) + 0.5) / nbin
    c2 = np.concatenate([counts, counts])
    mean = counts.mean()                    # αναμενόμενα γεγονότα ανά bin αν το προφίλ ήταν επίπεδο
    plt.axhspan(mean - np.sqrt(mean), mean + np.sqrt(mean), color='grey', alpha=0.15, zorder=0)
    plt.errorbar(phase, c2, yerr=np.sqrt(c2), drawstyle='steps-mid', label=label)
    plt.xlabel('phase'); plt.ylabel('counts per phase bin')

✏️ **TODO:**
1. Επιλέξτε τον αριθμό των bins φάσης `nbin` (δοκιμάστε 8, 16, 32).
2. Η αρχική μας εκτίμηση για την περίοδο είναι `P_guess = 31.6` s (από παλαιότερες παρατηρήσεις).
3. Δώστε στο `P_random` διάφορες τιμές και συγκρίνετε τα δύο προφίλ: π.χ. 25, 31, 31.5, 31.58, 31.62…

❓ Πόσο μακριά από την `P_guess` πρέπει να είναι η `P_random` ώστε το προφίλ της να φαίνεται **επίπεδο**; Συγκρίνετε τη διαφορά $|P_{random} - P_{guess}|$ με τον αριθμό $P^2/T_{obs}$ που τυπώνεται παρακάτω.
*Υπόδειξη:* κατά τη διάρκεια της παρατήρησης η πηγή κάνει $N = T_{obs}/P$ περιστροφές. Αν η δοκιμαστική περίοδος έχει λάθος $\Delta P$, η φάση των φωτονίων μετατοπίζεται κατά $N\,\Delta P/P = T_{obs}\,\Delta P/P^2$ κύκλους μέχρι το τέλος. Όταν η μετατόπιση γίνει ~1 κύκλος, ο παλμός «σβήνει».

In [ ]:
nbin = 8                 # TODO: αριθμός bins φάσης (8, 16, 32 ...)
P_guess = 31.6           # αρχική εκτίμηση της περιόδου (s)
P_random = 25.0          # TODO: δοκιμάστε διάφορες τιμές

plt.figure(figsize=(7, 4))
plot_profile(fold(t_ev, P_guess, nbin), label=f'P = {P_guess} s')
plot_profile(fold(t_ev, P_random, nbin), label=f'P = {P_random} s')
plt.legend()
plt.show()

print(f'P^2 / T_obs = {P_guess**2 / T_obs:.4f} s')

## 7. Αναζήτηση περιόδου σε ένα εύρος συχνοτήτων

Αντί να δοκιμάζουμε περιόδους μία-μία, σαρώνουμε πολλές δοκιμαστικές συχνότητες $f = 1/P$ και δίνουμε σε καθεμία ένα **σκορ**: πόσο απέχει το αναδιπλωμένο προφίλ από το επίπεδο. Ως σκορ χρησιμοποιούμε την **τυπική απόκλιση (rms)** των γεγονότων στα bins φάσης:
- επίπεδο προφίλ → μόνο τυχαίες (Poisson) διακυμάνσεις → μικρό rms·
- προφίλ με παλμό → μεγάλες διαφορές μεταξύ bins → μεγάλο rms.

Η καλύτερη συχνότητα είναι αυτή με το **μέγιστο** σκορ. (Αυτό είναι ισοδύναμο με το συνηθισμένο κριτήριο $\chi^2$ ως προς επίπεδο προφίλ.)

**Βήμα συχνότητας.** Δύο συχνότητες που διαφέρουν λιγότερο από $1/T_{obs}$ δίνουν σχεδόν το ίδιο προφίλ (η μετατόπιση φάσης είναι μικρότερη από έναν κύκλο, βλ. ενότητα 6). Άρα το $1/T_{obs}$ είναι η φυσική διακριτική ικανότητα. Χρησιμοποιούμε βήμα $df = 1/(\text{oversample} \times T_{obs})$, δηλαδή λίγα σημεία ανά στοιχείο διακριτικής ικανότητας.

In [ ]:
def period_search(times, freqs, nbin):
    # rms του αναδιπλωμένου προφίλ για κάθε δοκιμαστική συχνότητα
    score = np.zeros(len(freqs))
    for k in range(len(freqs)):
        f = freqs[k]
        phase = (times * f) % 1.0
        counts, _edges = np.histogram(phase, bins=nbin, range=(0.0, 1.0))
        score[k] = np.std(counts)
    return score

f_guess = 1 / P_guess
oversample = 5           # TODO: δοκιμάστε 1, 5, 0.2 και δείτε τι αλλάζει
df = 1 / (oversample * T_obs)
freqs = np.arange(f_guess - 1e-3, f_guess + 1e-3, df)

score_all = period_search(t_ev, freqs, nbin)

plt.figure(figsize=(8, 4))
plt.plot(freqs, score_all)
plt.axvline(f_guess, ls=':', color='grey', label='f_guess')
plt.xlabel('frequency (Hz)'); plt.ylabel('rms of folded profile (counts)')
plt.legend(); plt.show()
print('best frequency (all data):', freqs[np.argmax(score_all)], 'Hz')

❓ Το περιοδόγραμμα όλης της παρατήρησης έχει **πολλές** κορυφές και όχι μία οξεία γραμμή. Δύο λόγοι:
1. η συχνότητα του πάλσαρ **αλλάζει** κατά τη διάρκεια της παρατήρησης (αυτό ακριβώς θέλουμε να μετρήσουμε!)·
2. τα δεδομένα έχουν κανονικά **κενά** (ενότητα 5).

Άρα δεν μπορούμε να χρησιμοποιήσουμε όλα τα δεδομένα μαζί: θα τα χωρίσουμε σε μικρότερα τμήματα.

## 8. Επιλογή μέρους των δεδομένων: μάσκες και aliasing

Για να επιλέξουμε τα φωτόνια ενός χρονικού διαστήματος χρησιμοποιούμε μια **μάσκα**: έναν πίνακα με `True`/`False`, ένα για κάθε φωτόνιο. Παρακάτω κρατάμε μόνο τα πρώτα 30 ks και αναζητούμε την περίοδο εκεί.

**Πλευρικές κορυφές (aliasing).** Το περιοδόγραμμα ενός τμήματος δείχνει μια κύρια κορυφή και μικρότερες κορυφές δεξιά και αριστερά. Οφείλονται στα **κενά της τροχιάς**: τα δεδομένα «ανάβουν και σβήνουν» σε κάθε τροχιά του NuSTAR ($P_{orb} \approx 96$ min $\approx 5800$ s). Εξαιτίας αυτού του κανονικού «παραθύρου», το σήμα εμφανίζεται και σε συχνότητες $f \pm k/P_{orb}$, με $1/P_{orb} \approx 1.7\times10^{-4}$ Hz. Οι διακεκομμένες γραμμές δείχνουν τα $f_{peak} \pm 1/P_{orb}$. Θα το συζητήσουμε στον πίνακα.

In [ ]:
P_orb = 5800.0           # τροχιακή περίοδος NuSTAR (s)

mask1 = (t_ev >= 0) & (t_ev < 30e3)       # πρώτα 30 ks
t1 = t_ev[mask1]
freqs1 = np.arange(f_guess - 1e-3, f_guess + 1e-3, 1 / (oversample * (t1.max() - t1.min())))
score1 = period_search(t1, freqs1, nbin)
f1 = freqs1[np.argmax(score1)]

plt.figure(figsize=(8, 4))
plt.plot(freqs1, score1, label='0-30 ks')
for k in (-1, 1):
    plt.axvline(f1 + k / P_orb, ls='--', color='grey',label='alias')
plt.axvline(f1, ls=':', color='magenta',label='best')
plt.xlabel('frequency (Hz)'); plt.ylabel('rms of folded profile (counts)')
plt.legend(); plt.show()
print(f'0-30 ks: f = {f1:.7f} Hz,  P = {1/f1:.4f} s')

✏️ **TODO:** στο κενό κελί παρακάτω, κάντε το ίδιο για το διάστημα **270–300 ks**: φτιάξτε `mask2`, `t2`, `freqs2`, `score2`, `f2`, και σχεδιάστε τα `score1` και `score2` **στο ίδιο διάγραμμα** (αντιγράψτε και τροποποιήστε το παραπάνω κελί).

❓ Η κύρια κορυφή μετακινείται σε μεγαλύτερη ή μικρότερη συχνότητα; Πόσο; Από το $\Delta f$ και τον χρόνο μεταξύ των δύο διαστημάτων, κάντε μια πρώτη χονδρική εκτίμηση του $\dot\nu \approx \Delta f / \Delta t$. Το πάλσαρ επιταχύνεται ή επιβραδύνεται;

In [ ]:
# TODO: διάστημα 270-300 ks, σχεδιάστε μαζί με το 0-30 ks

## 9. Χωρισμός της παρατήρησης σε N τμήματα

Η παρακάτω συνάρτηση χωρίζει τα δεδομένα σε `n_seg` τμήματα ίσης διάρκειας, κάνει την αναζήτηση περιόδου σε καθένα και κρατά τη συχνότητα του μεγίστου.

✏️ **TODO:**
1. Επιλέξτε το `n_seg` (δοκιμάστε π.χ. 4, 8, 12) και τον αριθμό bins φάσης `nbin_seg`.
2. Κοιτάξτε το περιοδόγραμμα **κάθε** τμήματος (πλέγμα μικρών διαγραμμάτων). Η κόκκινη γραμμή δείχνει τη συχνότητα που επιλέχθηκε. Αν σε κάποιο τμήμα ο κώδικας διάλεξε **πλευρική κορυφή (alias)** αντί για την κύρια, γράψτε τον αριθμό του τμήματος στη λίστα `exclude` (π.χ. `exclude = [3]`) και τρέξτε ξανά το κελί.

❓ Ποιος είναι ο συμβιβασμός όταν επιλέγετε το `n_seg`; Περισσότερα τμήματα δίνουν περισσότερα σημεία για την προσαρμογή, αλλά κάθε τμήμα είναι μικρότερο: λιγότερα φωτόνια και πλατύτερη κορυφή (διακριτική ικανότητα $1/T_{seg}$).

**Σημειώστε** τα `n_seg` και `nbin_seg` που χρησιμοποιήσατε τελικά: μπαίνουν στο Google Sheet.

In [ ]:
def search_segments(t, n_seg, nbin, f_guess, oversample=5, half_width=1e-3):
    edges = np.linspace(t.min(), t.max(), n_seg + 1)
    T_mid, F_best, pgrams = [], [], []
    for k in range(n_seg):
        m = (t >= edges[k]) & (t < edges[k + 1])
        ts = t[m]
        fr = np.arange(f_guess - half_width, f_guess + half_width, 1 / (oversample * (ts.max() - ts.min())))
        sc = period_search(ts, fr, nbin)
        T_mid.append(ts.mean())
        F_best.append(fr[np.argmax(sc)])
        pgrams.append((fr, sc))
    return np.array(T_mid), np.array(F_best), pgrams

In [ ]:
n_seg = 1                # TODO: number of segments
nbin_seg = 10            # TODO: number of phase bins

T_mid, F_best, pgrams = search_segments(t_ev, n_seg, nbin_seg, f_guess)

ncol = 4
nrow = int(np.ceil(n_seg / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 2.6 * nrow), squeeze=False)
for k, (fr, sc) in enumerate(pgrams):
    ax = axes.flat[k]
    ax.plot(fr * 1e3, sc)
    ax.axvline(F_best[k] * 1e3, color='red')
    ax.set_title(f'segment {k}  (t = {T_mid[k]/1e3:.0f} ks)', fontsize=9)
for ax in axes.flat[n_seg:]:
    ax.axis('off')
fig.supxlabel('frequency (mHz)'); fig.supylabel('rms')
plt.tight_layout(); plt.show()

T_data, F_data = T_mid, F_best

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(T_data / 1e3, F_data * 1e3, 'o')
plt.xlabel('time since first photon (ks)'); plt.ylabel('frequency (mHz)')
plt.show()

## 10. Γραμμική προσαρμογή: $\nu(t) = \dot\nu\,(t - t_{ref}) + \nu_0$

Προσαρμόζουμε στα σημεία μια ευθεία $y = m\,(t - t_{ref}) + b$: η κλίση $m$ είναι ο $\dot\nu$ και το $b$ είναι η συχνότητα $\nu_0$ τη χρονική στιγμή αναφοράς $t_{ref}$. Επιλέγουμε το $t_{ref}$ στο μέσο της παρατήρησης.

<details><summary><b>Πώς λειτουργεί η προσαρμογή και τα σφάλματά της (προαιρετικό)</b></summary>

Η `scipy.optimize.curve_fit` βρίσκει τις παραμέτρους που ελαχιστοποιούν το άθροισμα των τετραγώνων των υπολοίπων
$$S(m,b) = \sum_i \left[y_i - \big(m\,(t_i - t_{ref}) + b\big)\right]^2 .$$
Επιστρέφει τις βέλτιστες τιμές και τον **πίνακα συνδιακύμανσης** `pcov`:
$$\mathrm{pcov} = \begin{pmatrix}\sigma_m^2 & \mathrm{cov}(m,b)\\ \mathrm{cov}(m,b) & \sigma_b^2\end{pmatrix}.$$
Τα διαγώνια στοιχεία είναι οι **διασπορές** των παραμέτρων, άρα τα σφάλματα είναι $\sigma_m = \sqrt{\mathrm{pcov}[0,0]}$ και $\sigma_b = \sqrt{\mathrm{pcov}[1,1]}$. Το μη διαγώνιο στοιχείο δείχνει πόσο συσχετίζονται οι δύο παράμετροι· επιλέγοντας το $t_{ref}$ στο μέσο των δεδομένων γίνεται σχεδόν μηδέν.

Επειδή δεν δίνουμε σφάλματα για τα επιμέρους σημεία, η `curve_fit` εκτιμά τη διασπορά των σημείων από τα υπόλοιπα, $s^2 = S_{min}/(N-2)$ ($N$ σημεία, 2 παράμετροι), και κλιμακώνει τα σφάλματα με αυτή. Έτσι τα σφάλματα δείχνουν πόσο καλά ακολουθούν τα σημεία μια ευθεία.
</details>

In [ ]:
t_ref = 0.5 * T_obs

def lin(t, m, b):
    return m * (t - t_ref) + b

fit_params, pcov = scipy.optimize.curve_fit(lin, T_data, F_data, p0=[0.0, f_guess])
m, b = fit_params
m_err, b_err = np.sqrt(np.diag(pcov))

F_fit = T_data*m +b #predicted Freq by best fit model

print(f'nudot = m = {m:.3e} +/- {m_err:.1e} Hz/s')
print(f'nu0   = b = {b:.7f} +/- {b_err:.1e} Hz   (at t_ref = {t_ref:.0f} s)')

## 11. Τελικό διάγραμμα (για την εργασία)

✏️ **TODO:**
1. Στο πάνω διάγραμμα σχεδιάστε την **ευθεία της προσαρμογής** με τα `m` και `b` (προσοχή στις μονάδες των αξόνων: χρόνος σε ks, συχνότητα σε mHz). Ο πίνακας `tt` περιέχει χρόνους για να σχεδιάσετε την ευθεία.
2. Υπολογίστε τα **υπόλοιπα** (δεδομένα − μοντέλο) για κάθε σημείο: αντικαταστήστε τη γραμμή `resid = np.zeros_like(F_fit)`.

Αυτό το διάγραμμα πρέπει να συμπεριληφθεί στην εργασία σας.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))

ax1.plot(T_data / 1e3, F_data * 1e3, 'o', label='segments')
tt = np.linspace(0, T_obs, 200)
# TODO: plot the best-fit line on ax1 (time in ks, frequency in mHz)
ax1.set_ylabel('frequency (mHz)')
ax1.legend()

# TODO: residuals = data - model, for each point
resid = np.zeros_like(F_fit) # <-TODO define residuals as (data − model)

ax2.plot(T_data / 1e3, resid * 1e6, 'o')
ax2.axhline(0, color='grey', lw=1)
ax2.set_xlabel('time since first photon (ks)')
ax2.set_ylabel(r'residual ($\mu$Hz)')
plt.show()

## 12. Αποτελέσματα για το κοινό Google Sheet

Η παράγωγος της περιόδου προκύπτει από το $P = 1/\nu$: $\dot P = -\dot\nu/\nu^2$, με σφάλμα $\sigma_{\dot P} = \sigma_{\dot\nu}/\nu^2$.

✏️ **TODO:** συμπληρώστε όνομα και αριθμό μητρώου και τρέξτε το κελί. Η τελευταία γραμμή είναι **χωρισμένη με tab**: αντιγράψτε την και επικολλήστε τη στην πρώτη κενή γραμμή του Google Sheet, και οι τιμές θα μπουν σε ξεχωριστές στήλες. Ελέγξτε ότι οι στήλες αντιστοιχούν στην επικεφαλίδα του sheet.

In [ ]:
student_name = ''        # TODO: your name
student_id = ''          # TODO: your student number

Pdot = -m / b**2
Pdot_err = m_err / b**2
print(f'P     = {1/b:.5f} s')
print(f'nudot = {m:.3e} +/- {m_err:.1e} Hz/s')
print(f'Pdot  = {Pdot:.3e} +/- {Pdot_err:.1e} s/s')
print()
print('name | student no. | file | n_seg | nbin | nudot | nudot_err | Pdot | Pdot_err')
print('\t'.join([student_name, student_id, my_choice, str(n_seg), str(nbin_seg),
                 f'{m:.4e}', f'{m_err:.2e}', f'{Pdot:.4e}', f'{Pdot_err:.2e}']))

---
# EXTRA A: μαγνητικό πεδίο και αβεβαιότητες (Μέρος 3 των σημειώσεων)

Από τις σημειώσεις, σχέση [7] (μονάδες cgs):
$$B_s = 2^{1/4}\,(2\pi)^{7/2}\; G^{-3/2}\, M^{-3/2}\, R^{-3}\, I^{7/2}\, \xi^{-7/4}\, \dot\nu^{7/2}\, \dot m^{-3},
\qquad \dot m = \frac{L R}{G M},\qquad L = 4\pi d^2 F.$$

✏️ **TODO:** δώστε την απόσταση που επιλέξατε στο Μέρος 2 και το σφάλμα της. Η ροή είναι η ευρυζωνική τιμή των Carpano et al. (2018), με σφάλμα 10%.

In [ ]:
G = 6.674e-8; Msun = 1.989e33; yr = 3.156e7; Mpc = 3.086e24      # cgs

def B_surface(nudot, mdot, M, R, I, xi):
    # σχέση [7] των σημειώσεων (cgs)
    return (2**0.25 * (2*np.pi)**3.5 * G**-1.5 * M**-1.5 * R**-3.0
            * I**3.5 * xi**-1.75 * nudot**3.5 * mdot**-3.0)

d_Mpc, d_err = 1.88, 0.05             # TODO: η απόστασή σας και το σφάλμα της (Mpc)
F, F_err = 1.1e-11, 0.11e-11          # erg/s/cm^2 (Carpano et al. 2018, 10%)
M, R, xi = 1.4 * Msun, 1.0e6, 0.5
I = 0.4 * M * R**2

L = 4 * np.pi * (d_Mpc * Mpc)**2 * F
mdot = L * R / (G * M)
B = B_surface(m, mdot, M, R, I, xi)
print(f'L = {L:.2e} erg/s,  mdot = {mdot:.2e} g/s = {mdot*yr/Msun:.2e} Msun/yr')
print(f'B = {B:.2e} G')

### Διάδοση σφαλμάτων

Για γινόμενο δυνάμεων, $B \propto \dot\nu^{7/2}\,\dot m^{-3}$, τα **σχετικά** σφάλματα προστίθενται τετραγωνικά, το καθένα πολλαπλασιασμένο με τον εκθέτη του:
$$\left(\frac{\sigma_B}{B}\right)^2 = \left(\frac{7}{2}\,\frac{\sigma_{\dot\nu}}{\dot\nu}\right)^2 + \left(3\,\frac{\sigma_{\dot m}}{\dot m}\right)^2,
\qquad
\left(\frac{\sigma_{\dot m}}{\dot m}\right)^2 = \left(2\,\frac{\sigma_d}{d}\right)^2 + \left(\frac{\sigma_F}{F}\right)^2 .$$

✏️ **TODO:** υπολογίστε τα `rel_mdot` και `rel_B` με αυτούς τους τύπους.

In [ ]:
rel_mdot = 0.0           # TODO: σχετικό σφάλμα του mdot
rel_B = 0.0              # TODO: σχετικό σφάλμα του B
print(f'B = {B:.2e} +/- {rel_B*B:.1e} G   ({100*rel_B:.0f}%)')

---
# EXTRA B: έχει σημασία εδώ η βαρυκεντρική διόρθωση;

Ο παρατηρητής κινείται: η Γη περιφέρεται γύρω από τον Ήλιο με ~30 km/s. Λόγω φαινομένου Doppler μετράμε $\nu_{obs} \simeq \nu\,(1 + v_\parallel/c)$, όπου $v_\parallel$ η ταχύτητα του παρατηρητή προς την πηγή. Αν η $v_\parallel$ αλλάζει κατά την παρατήρηση, μετράμε και μια ψευδή παράγωγο συχνότητας,
$$\dot\nu_{app} \simeq \frac{\nu}{c}\,\frac{dv_\parallel}{dt}.$$
Η βαρυκεντρική διόρθωση αφαιρεί αυτά τα φαινόμενα. Παρακάτω υπολογίζουμε με το `astropy` την πραγματική ταχύτητα της Γης προς την NGC 300 ULX-1 κατά τη διάρκεια της παρατήρησης.

❓ Είναι σημαντικό το φαινόμενο σε σχέση με τον $\dot\nu$ σας και το σφάλμα του; Συγκρίνετε με τα αποτελέσματα στο Google Sheet από αρχεία με και χωρίς διόρθωση. Για ποιο είδος ανάλυσης θα ήταν απαραίτητη η διόρθωση; (Υπόδειξη: ο χρόνος διάδοσης του φωτός αλλάζει κατά δεκάδες δευτερόλεπτα, περίπου μία περίοδο του παλμού.)

In [ ]:
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, get_body_barycentric_posvel

c_kms = 299792.458
source = SkyCoord(13.7703 * u.deg, -37.6955 * u.deg)   # NGC 300 ULX-1 / SN 2010da
n_hat = source.cartesian.xyz.value

t_grid = Time(hdr['DATE-OBS'], scale='utc') + np.linspace(0, T_obs, 200) * u.s
pos, vel = get_body_barycentric_posvel('earth', t_grid)
v_los = vel.xyz.to(u.km / u.s).value.T @ n_hat           # km/s
r_los = pos.xyz.to(u.km).value.T @ n_hat                 # km

nudot_app = b * (v_los[-1] - v_los[0]) / T_obs / c_kms
print(f'Earth velocity towards the source: {v_los[0]:.2f} -> {v_los[-1]:.2f} km/s')
print(f'apparent nudot: {nudot_app:.1e} Hz/s   (your nudot: {m:.2e} +/- {m_err:.1e} Hz/s)')
print(f'worst case (a_Earth along the line of sight): {b * 29.8**2 / 1.496e8 / c_kms:.1e} Hz/s')
print(f'change of light-travel time during the observation: {(r_los.max() - r_los.min()) / c_kms:.1f} s')

---
# EXTRA C: σύγκριση με το Stingray

Το [Stingray](https://docs.stingray.science) είναι πακέτο Python για χρονική ανάλυση στις ακτίνες Χ. Περιέχει έτοιμες εκδοχές όσων γράψαμε μόνοι μας: το `epoch_folding_search` (η εκδοχή $\chi^2$ του δικού μας σκορ rms) και το `z_n_search` (η στατιστική $Z^2_n$, Buccheri et al. 1983, που είναι ευαίσθητη σε προφίλ που αποτελούνται από λίγες ημιτονοειδείς αρμονικές). Παρακάτω τα συγκρίνουμε με τη δική μας `period_search` στα πρώτα 30 ks.

In [ ]:
try:
    from stingray.pulse.search import epoch_folding_search, z_n_search
except ImportError:
    print('stingray is not installed: conda install -c conda-forge stingray')
else:
    _, ef = epoch_folding_search(t1, freqs1, nbin=nbin)
    _, zn = z_n_search(t1, freqs1, nbin=40, nharm=2)
    plt.figure(figsize=(8, 4))
    plt.plot(freqs1, score1 / score1.max(), label='our rms score')
    plt.plot(freqs1, ef / ef.max(), ls='--', label='stingray epoch folding')
    plt.plot(freqs1, zn / zn.max(), ls=':', label=r'stingray $Z^2_2$')
    plt.xlabel('frequency (Hz)'); plt.ylabel('normalised statistic'); plt.legend()
    plt.show()